In [1]:
#setuop
!pip install telethon kafka-python-ng

import json
import asyncio
from datetime import datetime
from telethon import TelegramClient, events
from kafka import KafkaProducer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

In [ ]:
#config
API_ID = 12345678              # Your API ID
API_HASH = "your_api_hash_here" # Your API Hash
CHANNELS = ["newsonlineils", "koahadasotbatelegram", "danielamram3"]
TOPIC = "telegram-raw"

# 2. Connect to Kafka
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8')
)

# 3. Connect to Telegram
client = TelegramClient("tele_session", API_ID, API_HASH)

# 4. Define what happens when a message arrives
@client.on(events.NewMessage(chats=CHANNELS))
async def handle_new_message(event):
    if not event.raw_text:
        return

    payload = {
        "channel": str(event.chat_id),
        "text": event.raw_text,
        "ts": int(event.date.timestamp())
    }

    producer.send(TOPIC, value=payload)
    print(f"Sent to Kafka: {event.raw_text[:40]}...")

# 5. Start listening
await client.start()
print("Listening to Telegram channels...")
await client.run_until_disconnected()